# SOXL paper/sim replay production candidate v1

目标：把已经验证的 `schedule -> dashboard -> paper intent` 链路固定成 production 候选 notebook。

边界：v1 只运行 paper/sim，不连接真实 broker，不触发真实下单。生产晋升前仍需更严格模型、风控和外部调度。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.api import (
    PaperBrokerAdapter,
    build_target_position_intent_from_dashboard,
    execute_paper_intent,
    write_intent_ledger,
)
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('production/paper_sim_replay_v1')
CARD.mkdir(parents=True, exist_ok=True)
notifications = []
config = ModelDashboardPipelineConfig(
    name='production_paper_sim_replay_v1',
    card_dir=CARD / 'matrix_card',
    dashboard_output_path=CARD / 'dashboard_export.json',
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'random_forest_clf'),
    top_n_factors=3,
    dashboard_top_n=5,
    retries=1,
)
config

ModelDashboardPipelineConfig(name='production_paper_sim_replay_v1', card_dir=PosixPath('production/paper_sim_replay_v1/matrix_card'), dashboard_output_path=PosixPath('production/paper_sim_replay_v1/dashboard_export.json'), start='2026-01-01', end='2026-06-09', train_end='2026-04-30', model_names=('ridge_mse', 'random_forest_clf'), top_n_factors=3, dashboard_top_n=5, retries=1)

In [2]:
schedule_result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / 'locks',
    notify=notifications.append,
)
schedule_result

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


JobResult(name='production_paper_sim_replay_v1', status='success', payload={'matrix_card': 'production/paper_sim_replay_v1/matrix_card', 'dashboard_export': 'production/paper_sim_replay_v1/dashboard_export.json', 'dashboard_schema_version': 'dashboard.model_matrix.v1', 'top_model': 'random_forest_clf__all_features', 'matrix_status': 'feature_set_matrix_completed', 'variant_count': 6, 'attempts': 1}, detail='')

In [3]:
dashboard_payload = json.loads((CARD / 'dashboard_export.json').read_text(encoding='utf-8'))
intent = build_target_position_intent_from_dashboard(
    dashboard_payload,
    instrument='SOXL',
    min_position=0.0,
    max_position=0.75,
)
intent.to_dict()

{'instrument': 'SOXL',
 'target_position': 0.7407635422717518,
 'source_model': 'random_forest_clf__all_features',
 'source_artifact': 'matrix_card',
 'mode': 'paper',
 'schema_version': 'api.execution_intent.v1',
 'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
 'metadata': {'raw_avg_position': 0.7407635422717518,
  'min_position': 0.0,
  'max_position': 0.75,
  'base_model': 'random_forest_clf',
  'feature_set': 'all_features',
  'stability_score': 54.214257763652995,
  'max_drawdown_pct': 21.677243377022705,
  'dashboard_schema_version': 'dashboard.model_matrix.v1'}}

In [4]:
broker = PaperBrokerAdapter(cash=10000.0, shares=0)
execution_result = execute_paper_intent(intent, broker=broker, price=100.0)
(CARD / 'execution_intent.json').write_text(json.dumps(intent.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
(CARD / 'paper_execution_result.json').write_text(json.dumps(execution_result.to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')
write_intent_ledger(CARD / 'intent_ledger.json', intent=intent, result=execution_result)
execution_result.to_dict()

{'schema_version': 'api.paper_execution_result.v1',
 'status': 'paper_executed',
 'broker_name': 'paper',
 'intent': {'instrument': 'SOXL',
  'target_position': 0.7407635422717518,
  'source_model': 'random_forest_clf__all_features',
  'source_artifact': 'matrix_card',
  'mode': 'paper',
  'schema_version': 'api.execution_intent.v1',
  'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
  'metadata': {'raw_avg_position': 0.7407635422717518,
   'min_position': 0.0,
   'max_position': 0.75,
   'base_model': 'random_forest_clf',
   'feature_set': 'all_features',
   'stability_score': 54.214257763652995,
   'max_drawdown_pct': 21.677243377022705,
   'dashboard_schema_version': 'dashboard.model_matrix.v1'}},
 'snapshot': {'cash': 2600.0, 'shares': 74, 'price': 100.0}}

In [5]:
summary = {
    'schema_version': 'production.paper_sim_replay.v1',
    'status': 'paper_replay_completed',
    'mode': intent.mode,
    'live_broker_enabled': False,
    'instrument': intent.instrument,
    'source_model': intent.source_model,
    'target_position': intent.target_position,
    'paper_shares': execution_result.snapshot.shares,
    'paper_position': execution_result.snapshot.position,
    'schedule_status': schedule_result.status,
    'schedule_attempts': schedule_result.payload.get('attempts'),
    'notifications': notifications,
    'dashboard_export': str(CARD / 'dashboard_export.json'),
    'intent_ledger': str(CARD / 'intent_ledger.json'),
    'matrix_card': str(CARD / 'matrix_card'),
    'guardrail': intent.guardrail,
}
(CARD / 'production_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'production.paper_sim_replay.v1',
 'status': 'paper_replay_completed',
 'mode': 'paper',
 'live_broker_enabled': False,
 'instrument': 'SOXL',
 'source_model': 'random_forest_clf__all_features',
 'target_position': 0.7407635422717518,
 'paper_shares': 74,
 'paper_position': 0.74,
 'schedule_status': 'success',
 'schedule_attempts': 1,
 'notifications': ['production_paper_sim_replay_v1:success'],
 'dashboard_export': 'production/paper_sim_replay_v1/dashboard_export.json',
 'intent_ledger': 'production/paper_sim_replay_v1/intent_ledger.json',
 'matrix_card': 'production/paper_sim_replay_v1/matrix_card',
 'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。'}

In [6]:
{
    'dashboard_top_models': dashboard_payload['top_models'][:3],
    'feature_set_summary': dashboard_payload['feature_set_summary'],
    'ledger_exists': (CARD / 'intent_ledger.json').exists(),
    'live_broker_enabled': summary['live_broker_enabled'],
}

{'dashboard_top_models': [{'rank': 1,
   'model': 'random_forest_clf__all_features',
   'base_model': 'random_forest_clf',
   'feature_set': 'all_features',
   'return_pct': 49.032787769965715,
   'benchmark_return_pct': 62.1472468532841,
   'relative_return_pct': -13.114459083318389,
   'max_drawdown_pct': 21.677243377022705,
   'sharpe': 3.8534654423822032,
   'avg_position': 0.7407635422717518,
   'turnover': 3.1283151044281827,
   'stability_score': 54.214257763652995},
  {'rank': 2,
   'model': 'ridge_mse__top_ic_factors',
   'base_model': 'ridge_mse',
   'feature_set': 'top_ic_factors',
   'return_pct': 51.68889425463847,
   'benchmark_return_pct': 62.1472468532841,
   'relative_return_pct': -10.458352598645632,
   'max_drawdown_pct': 26.17632325678787,
   'sharpe': 3.8153069361698657,
   'avg_position': 0.7496923542352576,
   'turnover': 2.1228598727860692,
   'stability_score': 52.875839799780145},
  {'rank': 3,
   'model': 'ridge_mse__ic_weighted_composite',
   'base_model': '

## 中文结论

- 做了什么：运行 production 候选链路，完成 schedule 刷新、dashboard export、paper intent 生成和 PaperBrokerAdapter 模拟执行。
- 核心指标：source model=`random_forest_clf__all_features`，target_position=0.7408，paper shares=74，paper position=0.7400。
- regime 差异：dashboard export 保留 `regime_metrics`、`qlib_risk_metrics` 和 `model_effect_deltas`，生产回放通过这些 sidecar review。
- 是否晋升：否。本 notebook 是 paper/sim production candidate，不是 live 生产策略。
- 风险：`live_broker_enabled=False`；guardrail=`v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。`。后续需要严格模型实现、风控阈值、外部调度和审计后才能讨论 live adapter。
- 下一步：补 manifest 化生产注册表和更严格的模型 adapter 替换计划。